In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


DATA_ROOT = r'C:\Users\omarf\Downloads\archive (3)\mnist-original.mat'

from torchvision import datasets, transforms
from scipy.io import loadmat
import numpy as np

transform = transforms.Compose([
    transforms.ToTensor(),                         
    transforms.Normalize((0.1307,), (0.3081,))       
])

data = loadmat(DATA_ROOT)
X, y = data['data'].T, data['label'].squeeze()  

X_train, y_train = X[:60000], y[:60000]
X_test, y_test = X[60000:], y[60000:]

train_set = TensorDataset(
    torch.from_numpy(X_train.reshape(-1, 1, 28, 28)).float() / 255.0,
    torch.from_numpy(y_train.astype(np.int64))
)
test_set = TensorDataset(
    torch.from_numpy(X_test.reshape(-1, 1, 28, 28)).float() / 255.0,
    torch.from_numpy(y_test.astype(np.int64))
)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=128, shuffle=False)

print(f"Train samples: {len(train_set):,} | Test samples: {len(test_set):,}")

# ---------------------------------------------------------------------------
# 2. MODEL ARCHITECTURE (matches the blueprint: 2 conv blocks, lean head)
# ---------------------------------------------------------------------------
class MNIST_CNN(nn.Module):
    """
      Input:            1 x 28 x 28
      Conv Block 1:      32 filters, 3x3, padding=1 -> ReLU -> MaxPool(2)
                         -> 32 x 14 x 14
      Conv Block 2:      64 filters, 3x3, padding=1 -> ReLU -> MaxPool(2)
                         -> 64 x 7 x 7
      Flatten:           64*7*7 = 3136
      Dense:             3136 -> 128 -> ReLU -> Dropout
      Output:            128 -> 10 (class logits)
    """
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x, trace_shapes=False):
        if trace_shapes: print(f"input:        {tuple(x.shape)}")

        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        if trace_shapes: print(f"after block1: {tuple(x.shape)}")   

        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        if trace_shapes: print(f"after block2: {tuple(x.shape)}")  

        x = torch.flatten(x, start_dim=1)
        if trace_shapes: print(f"flattened:    {tuple(x.shape)}") 

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        if trace_shapes: print(f"logits:       {tuple(x.shape)}")  

        return x


model = MNIST_CNN(num_classes=10).to(device)
print(model)
print(f"\nTotal trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

print("\n--- Shape trace through the network ---")
with torch.no_grad():
    dummy = torch.randn(4, 1, 28, 28).to(device)
    _ = model(dummy, trace_shapes=True)

# ---------------------------------------------------------------------------
# 3. LOSS + OPTIMIZER
# ---------------------------------------------------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ---------------------------------------------------------------------------
# 4. TRAINING LOOP
# ---------------------------------------------------------------------------
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, num_classes=10):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    confusion = torch.zeros(num_classes, num_classes, dtype=torch.int64)

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        for t, p in zip(labels.view(-1), preds.view(-1)):
            confusion[t.long(), p.long()] += 1

    return running_loss / total, correct / total, confusion


print("\n--- Training ---")
EPOCHS = 8
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    test_loss, test_acc, confusion = evaluate(model, test_loader, criterion)
    print(f"Epoch {epoch:2d}/{EPOCHS} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
          f"test_loss={test_loss:.4f} test_acc={test_acc:.3f}")


# ---------------------------------------------------------------------------
# 5. CONFUSION MATRIX (rows = true label, cols = predicted label)
# ---------------------------------------------------------------------------
print("\n--- Confusion matrix (final epoch, test set) ---")
header = "     " + " ".join(f"{i:4d}" for i in range(10))
print(header)
for i, row in enumerate(confusion.tolist()):
    print(f"{i:3d}  " + " ".join(f"{v:4d}" for v in row))


Using device: cpu
Train samples: 60,000 | Test samples: 10,000
MNIST_CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

Total trainable parameters: 421,834

--- Shape trace through the network ---
input:        (4, 1, 28, 28)
after block1: (4, 32, 14, 14)
after block2: (4, 64, 7, 7)
flattened:    (4, 3136)
logits:       (4, 10)

--- Training ---
Epoch  1/8 | train_loss=0.1568 train_acc=0.951 | test_loss=0.0450 test_acc=0.986
Epoch  2/